# 07 - Scaling: Generate COTs

Generate COTs for Qwen3-1.7B, Qwen3-8B, and Qwen3-14B (Qwen3-4B already done in 02).
This enables measuring how legibility scales with model size within a family.

In [ ]:
import subprocess, sys
from pathlib import Path

WORKSPACE = Path("/workspace/13-4-2026")
REPO_DIR = WORKSPACE / "legibility"

if REPO_DIR.exists():
    subprocess.run(["git", "-C", str(REPO_DIR), "pull"], check=True)
else:
    WORKSPACE.mkdir(parents=True, exist_ok=True)
    subprocess.run([
        "git", "clone", "-b", "13-4-2026",
        "https://github.com/JackHopkins/legibility.git",
        str(REPO_DIR)
    ], check=True)

sys.path.insert(0, str(REPO_DIR))
from lib.config import *

for d in [CACHE_DIR, COT_CACHE, PARAPHRASE_CACHE, PREFILL_CACHE, RESULTS_DIR, FIGURES_DIR]:
    d.mkdir(parents=True, exist_ok=True)

In [ ]:
import json
import re
from tqdm import tqdm
from vllm import LLM, SamplingParams
from transformers import AutoTokenizer
from lib.data import load_gsm8k, extract_predicted_answer
from lib.prompts import build_cot_messages, build_no_cot_messages

dataset = load_gsm8k()
print(f"Loaded {len(dataset)} GSM8K problems")

# Models to generate COTs for (skip 4B, already done)
scaling_models = [m for m in SCALING_MODELS if m != PRIMARY_MODEL]
print(f"Will generate COTs for: {scaling_models}")

In [ ]:
import gc, torch

def model_tag(model_name):
    """Short tag for cache directories, e.g. 'qwen3_1.7b'"""
    name = model_name.split("/")[-1].lower().replace("-", "_")
    return name

for model_name in scaling_models:
    tag = model_tag(model_name)
    cot_dir = CACHE_DIR / f"cots_{tag}"
    cot_dir.mkdir(parents=True, exist_ok=True)
    prefill_dir = PREFILL_CACHE  # no_cot results go in common prefill cache

    done_ids = {int(p.stem) for p in cot_dir.glob("*.json")}
    todo = [ex for ex in dataset if ex["problem_id"] not in done_ids]
    print(f"\n=== {model_name} ({tag}) ===")
    print(f"COTs: {len(done_ids)} done, {len(todo)} remaining")

    if not todo:
        print("Skipping - all done.")
        continue

    print(f"Loading {model_name}...")
    try:
        llm = LLM(model=model_name, dtype="bfloat16", max_model_len=4096)
    except Exception as e:
        print(f"Failed to load {model_name}: {e}")
        continue

    tokenizer = AutoTokenizer.from_pretrained(model_name)
    sampling_cot = SamplingParams(temperature=TEMPERATURE, max_tokens=MAX_COT_TOKENS)

    # Generate COTs
    for i in tqdm(range(0, len(todo), CHUNK_SIZE), desc=f"COTs ({tag})"):
        chunk = todo[i:i + CHUNK_SIZE]
        prompts = [
            tokenizer.apply_chat_template(
                build_cot_messages(ex["question"]),
                tokenize=False, add_generation_prompt=True,
                enable_thinking=True
            ) for ex in chunk
        ]
        outputs = llm.generate(prompts, sampling_cot)

        for ex, output in zip(chunk, outputs):
            full_response = output.outputs[0].text
            think_match = re.search(r"<think>(.*?)</think>", full_response, re.DOTALL)
            if think_match:
                cot_text = think_match.group(1).strip()
                answer_portion = full_response[think_match.end():].strip()
            else:
                parts = re.split(r"The answer is:?", full_response, maxsplit=1)
                cot_text = parts[0].strip()
                answer_portion = parts[1].strip() if len(parts) > 1 else full_response

            predicted = extract_predicted_answer(answer_portion)
            if predicted is None:
                predicted = extract_predicted_answer(full_response)

            result = {
                "problem_id": ex["problem_id"],
                "question": ex["question"],
                "gold_answer": ex["gold_answer"],
                "cot_text": cot_text,
                "predicted_answer": predicted,
                "full_response": full_response,
                "model": model_name,
            }
            (cot_dir / f"{ex['problem_id']}.json").write_text(json.dumps(result))

    # Also run no_cot for this model
    no_cot_cond = f"no_cot_{tag}"
    nc_done = {int(p.stem.split("_", 2)[-1]) for p in PREFILL_CACHE.glob(f"{no_cot_cond}_*.json")}
    nc_todo = [ex for ex in dataset if ex["problem_id"] not in nc_done]
    print(f"No-COT ({tag}): {len(nc_done)} done, {len(nc_todo)} remaining")

    sampling_short = SamplingParams(temperature=TEMPERATURE, max_tokens=MAX_ANSWER_TOKENS)
    for i in tqdm(range(0, len(nc_todo), CHUNK_SIZE), desc=f"No-COT ({tag})"):
        chunk = nc_todo[i:i + CHUNK_SIZE]
        prompts = [
            tokenizer.apply_chat_template(
                build_no_cot_messages(ex["question"]),
                tokenize=False, add_generation_prompt=True
            ) for ex in chunk
        ]
        outputs = llm.generate(prompts, sampling_short)
        for ex, output in zip(chunk, outputs):
            generated = output.outputs[0].text.strip()
            result = {
                "problem_id": ex["problem_id"],
                "condition": no_cot_cond,
                "model_used": model_name,
                "prefill_text": None,
                "predicted_answer": extract_predicted_answer(generated),
                "gold_answer": ex["gold_answer"],
                "generated_tokens": generated,
                "error": None,
            }
            (PREFILL_CACHE / f"{no_cot_cond}_{ex['problem_id']}.json").write_text(json.dumps(result))

    del llm, tokenizer
    gc.collect()
    torch.cuda.empty_cache()
    print(f"{model_name} complete and unloaded.")

In [ ]:
# Summary
for model_name in SCALING_MODELS:
    tag = model_tag(model_name)
    if model_name == PRIMARY_MODEL:
        n = len(list(COT_CACHE.glob("*.json")))
    else:
        n = len(list((CACHE_DIR / f"cots_{tag}").glob("*.json")))
    print(f"{model_name}: {n} COTs cached")